In [ ]:
import os
import numpy as np
import pandas as pd
import folium
import branca
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import math
import warnings
from datetime import datetime
import gc  # For garbage collection

warnings.filterwarnings('ignore')

# Use non-interactive backend for matplotlib
import matplotlib
matplotlib.use('Agg')

###########################################
# Configuration Parameters
###########################################

RUNTIME_CONFIG = {
    'DATA_FILES_PATTERN': 'JC2024*citibiketripdata.csv',
    'DATA_DIR': 'data/citibike',
    'OUTPUT_DIR': 'results',
    'MODE': 'all',  # 'train', 'visualize', 'all'
    'MAX_STATIONS': 50,  # Back to 50 stations
    'SAMPLE_RATE': 1.0,  # Full data
    'DISTANCE_THRESHOLD': 2.0,
    'ANALYZE_SEASONAL_PATTERNS': True,
    'CHUNK_SIZE': 10000,  # Process data in chunks
}

# Citibike time period configurations
TIME_PERIODS = {
    "Morning Peak (7-10 AM)": (7, 10),
    "Afternoon (10 AM-4 PM)": (10, 16),
    "Evening Peak (4-7 PM)": (16, 19),
    "All Day": (0, 24)
}

###########################################
# Helper Functions
###########################################

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in kilometers"""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371
    
    return c * r

def create_bezier_curve_with_strength(lon1, lat1, lon2, lat2, curve_strength=0.2, num_points=20):
    """Create smooth bezier curve between two points with customizable curve strength"""
    mid_lon = (lon1 + lon2) / 2
    mid_lat = (lat1 + lat2) / 2
    
    dx = lon2 - lon1
    dy = lat2 - lat1
    
    control_lon = mid_lon - dy * curve_strength
    control_lat = mid_lat + dx * curve_strength
    
    t = np.linspace(0, 1, num_points)
    curve = []
    for ti in t:
        x = (1-ti)**2 * lon1 + 2*(1-ti)*ti * control_lon + ti**2 * lon2
        y = (1-ti)**2 * lat1 + 2*(1-ti)*ti * control_lat + ti**2 * lat2
        curve.append([y, x])
    
    return curve

def create_bezier_curve(lon1, lat1, lon2, lat2, num_points=20):
    """Create smooth bezier curve between two points with dynamic strength"""
    dx = lon2 - lon1
    dy = lat2 - lat1
    distance = math.sqrt(dx**2 + dy**2)
    
    curve_strength = min(0.3, 0.05 + 0.1 * (distance / 0.05))
    
    return create_bezier_curve_with_strength(lon1, lat1, lon2, lat2, curve_strength, num_points)

###########################################
# Memory-Efficient Data Loading
###########################################

def load_citibike_data(data_dir, data_files_pattern, sample_rate=1.0, chunk_size=10000):
    """Load and preprocess Citibike data from multiple files with memory management"""
    print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
    file_pattern = os.path.join(data_dir, data_files_pattern)
    file_list = glob.glob(file_pattern)
    
    if not file_list:
        raise FileNotFoundError(f"No data files found matching pattern: {file_pattern}")
    
    file_list.sort()
    print(f"Found {len(file_list)} files to process")
    
    all_trips = []
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        try:
            month = int(file_name[6:8])
            print(f"Extracted month: {month}")
        except (ValueError, IndexError):
            month = 0
            print(f"Could not extract month from filename: {file_name}, using placeholder")
        
        # Process file in chunks to manage memory
        chunks = []
        chunk_count = 0
        
        try:
            for chunk in pd.read_csv(file_path, chunksize=chunk_size):
                # Sample the chunk if needed
                if sample_rate < 1.0:
                    sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
                else:
                    sampled_chunk = chunk
                
                chunks.append(sampled_chunk)
                chunk_count += 1
                
                # Print progress every 10 chunks
                if chunk_count % 10 == 0:
                    print(f"  Processed {chunk_count} chunks...")
        
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            continue
        
        if chunks:
            monthly_trips = pd.concat(chunks, ignore_index=True)
            print(f"Loaded {len(monthly_trips)} trips from {file_name}")
            
            if 'month' not in monthly_trips.columns:
                monthly_trips['month'] = month
            
            all_trips.append(monthly_trips)
            
            # Clear chunks from memory
            del chunks
            gc.collect()
        else:
            print(f"No data loaded from {file_name}")
    
    if not all_trips:
        raise ValueError("No trip data was loaded from any file")
    
    print("Combining all monthly data...")
    trips_df = pd.concat(all_trips, ignore_index=True)
    print(f"Combined dataset contains {len(trips_df)} trips")
    
    # Clear intermediate data
    del all_trips
    gc.collect()
    
    # Basic data cleaning
    print("Cleaning data...")
    required_cols = ['start_station_name', 'end_station_name', 'start_lat', 'start_lng', 
                    'end_lat', 'end_lng', 'started_at', 'ended_at']
    
    # Check which columns exist
    existing_cols = [col for col in required_cols if col in trips_df.columns]
    trips_df = trips_df.dropna(subset=existing_cols)
    
    print("Converting timestamps to datetime...")
    trips_df['started_at'] = pd.to_datetime(trips_df['started_at'], errors='coerce')
    trips_df['ended_at'] = pd.to_datetime(trips_df['ended_at'], errors='coerce')
    
    # Extract time features
    trips_df['start_hour'] = trips_df['started_at'].dt.hour
    trips_df['start_day'] = trips_df['started_at'].dt.dayofweek
    trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    if 'month' not in trips_df.columns:
        trips_df['month'] = trips_df['started_at'].dt.month
    
    # Calculate trip duration
    trips_df['duration_minutes'] = (trips_df['ended_at'] - trips_df['started_at']).dt.total_seconds() / 60
    
    # Filter out unrealistic trips
    trips_df = trips_df[(trips_df['duration_minutes'] > 0) & 
                        (trips_df['duration_minutes'] < 24*60)]
    
    print(f"After cleaning: {len(trips_df)} trips")
    
    return trips_df

def get_top_stations(trips_df, max_stations):
    """Get the top stations by trip count and create station mapping"""
    print("Calculating station popularity...")
    start_counts = trips_df['start_station_name'].value_counts()
    end_counts = trips_df['end_station_name'].value_counts()
    combined_counts = start_counts.add(end_counts, fill_value=0)
    
    max_stations = min(max_stations, len(combined_counts))
    top_stations = combined_counts.sort_values(ascending=False).head(max_stations).index.tolist()
    
    print(f"Using top {len(top_stations)} stations")
    
    station_mapping = {station: idx for idx, station in enumerate(top_stations)}
    
    station_data = {}
    
    for station in top_stations:
        start_rows = trips_df[trips_df['start_station_name'] == station]
        end_rows = trips_df[trips_df['end_station_name'] == station]
        
        if not start_rows.empty:
            lat = start_rows['start_lat'].iloc[0]
            lng = start_rows['start_lng'].iloc[0]
        elif not end_rows.empty:
            lat = end_rows['end_lat'].iloc[0]
            lng = end_rows['end_lng'].iloc[0]
        else:
            lat, lng = 0, 0
            print(f"No coordinate data found for station: {station}")
        
        popularity = (
            trips_df['start_station_name'].value_counts().get(station, 0) +
            trips_df['end_station_name'].value_counts().get(station, 0)
        )
        
        station_data[station_mapping[station]] = {
            'name': station,
            'lat': lat,
            'lng': lng,
            'popularity': popularity
        }
    
    # Create a dataframe from station_data for easy access
    stations_df = pd.DataFrame([
        {
            'name': data['name'],
            'lat': data['lat'],
            'lng': data['lng'],
            'popularity': data['popularity']
        }
        for idx, data in station_data.items()
    ])
    
    return station_mapping, station_data, top_stations, stations_df

def calculate_od_matrix(trips_df, station_mapping):
    """Calculate the origin-destination matrix from trips"""
    print("Calculating OD matrix...")
    mapped_trips = trips_df[
        trips_df['start_station_name'].isin(station_mapping.keys()) & 
        trips_df['end_station_name'].isin(station_mapping.keys())
    ]
    
    num_stations = len(station_mapping)
    od_matrix = pd.DataFrame(
        np.zeros((num_stations, num_stations)),
        index=range(num_stations),
        columns=range(num_stations)
    )
    
    for _, trip in mapped_trips.iterrows():
        origin_idx = station_mapping[trip['start_station_name']]
        dest_idx = station_mapping[trip['end_station_name']]
        od_matrix.iloc[origin_idx, dest_idx] += 1
    
    return od_matrix

###########################################
# Folium Visualization Functions
###########################################

def create_interactive_station_map(stations_df, map_title="Bike Stations", output_dir="results"):
    """Create an interactive map of bike stations with multiple layer options and controls"""
    print("Creating interactive station map...")
    
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,
        control_scale=True
    )
    
    # Add multiple tile layers
    folium.TileLayer('cartodbpositron', name='CartoDB Positron', control=True).add_to(m)
    folium.TileLayer('openstreetmap', name='OpenStreetMap', control=True).add_to(m)
    folium.TileLayer('cartodbdark_matter', name='CartoDB Dark Matter', control=True).add_to(m)
    
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satellite",
        control=True
    ).add_to(m)
    
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Standard",
        control=True
    ).add_to(m)
    
    # Create feature groups
    stations_group = folium.FeatureGroup(name="All Stations", show=True)
    popular_group = folium.FeatureGroup(name="Popular Stations", show=False)
    clusters_group = MarkerCluster(name="Clustered Stations")
    
    popularity_threshold = stations_df['popularity'].quantile(0.75)
    
    max_popularity = stations_df['popularity'].max()
    colormap = branca.colormap.LinearColormap(
        colors=['green', 'yellow', 'orange', 'red'],
        vmin=0,
        vmax=max_popularity,
        caption='Station Popularity'
    )
    
    colormap.add_to(m)
    
    for _, station in stations_df.iterrows():
        popup_html = f"""
        <strong>{station['name']}</strong><br>
        <strong>Popularity:</strong> {station['popularity']}<br>
        <strong>Location:</strong> {station['lat']:.4f}, {station['lng']:.4f}
        """
        
        color = colormap(station['popularity'])
        
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=5,
            color='black',
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(stations_group)
        
        if station['popularity'] >= popularity_threshold:
            folium.CircleMarker(
                location=[station['lat'], station['lng']],
                radius=8,
                color='black',
                fill=True,
                fill_color='red',
                fill_opacity=0.9,
                popup=folium.Popup(popup_html, max_width=300)
            ).add_to(popular_group)
        
        folium.Marker(
            location=[station['lat'], station['lng']],
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(clusters_group)
    
    stations_group.add_to(m)
    popular_group.add_to(m)
    clusters_group.add_to(m)
    
    folium.LayerControl().add_to(m)
    
    os.makedirs(output_dir, exist_ok=True)
    map_filename = os.path.join(output_dir, "interactive_station_map.html")
    m.save(map_filename)
    print(f"Interactive station map saved to {map_filename}")
    
    return m

def create_flow_map(od_matrix, stations_df, title="Bike Flow Map", flow_scale=1.0, min_trips=5):
    """Create an interactive flow map showing trip patterns between stations"""
    print(f"Creating flow map: {title}")
    
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,
        control_scale=True
    )
    
    folium.TileLayer('cartodbpositron', name='CartoDB Positron', control=True).add_to(m)
    folium.TileLayer('openstreetmap', name='OpenStreetMap', control=True).add_to(m)
    folium.TileLayer('cartodbdark_matter', name='CartoDB Dark Matter', control=True).add_to(m)
    
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satellite",
        control=True
    ).add_to(m)
    
    title_html = f'''
        <h3 align="center" style="font-size:16px"><b>{title}</b></h3>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    major_flows = folium.FeatureGroup(name="Major Flows (Top 20%)", show=True)
    medium_flows = folium.FeatureGroup(name="Medium Flows", show=True)
    minor_flows = folium.FeatureGroup(name="Minor Flows", show=False)
    
    max_flow = od_matrix.max().max()
    
    all_flows = od_matrix.values.flatten()
    all_flows = all_flows[all_flows >= min_trips]
    
    if len(all_flows) > 0:
        top_threshold = np.percentile(all_flows, 80)
        medium_threshold = np.percentile(all_flows, 50)
    else:
        top_threshold = max_flow
        medium_threshold = max_flow / 2
    
    colormap = branca.colormap.LinearColormap(
        colors=['blue', 'green', 'yellow', 'orange', 'red'],
        vmin=min_trips,
        vmax=max_flow,
        caption='Trip Volume'
    )
    
    colormap = colormap.add_to(m)
    
    legend_style = """
    <style>
        .legend {
            background-color: rgba(255, 255, 255, 0.9) !important;
            border-radius: 5px !important;
            padding: 6px !important;
            box-shadow: 0 0 5px rgba(0, 0, 0, 0.2) !important;
        }
    </style>
    """
    m.get_root().html.add_child(folium.Element(legend_style))
    
    station_colors = {}
    for i, station in enumerate(stations_df['name']):
        hue = (i * 30) % 360
        station_colors[station] = hue
    
    flow_count = 0
    for i in range(len(od_matrix)):
        for j in range(len(od_matrix)):
            if i != j and od_matrix.iloc[i, j] >= min_trips:
                start_station = stations_df.iloc[i]
                end_station = stations_df.iloc[j]
                
                flow = od_matrix.iloc[i, j]
                
                rel_flow = flow / max_flow
                
                width = 1 + (rel_flow)**1.2 * flow_scale * 8
                
                opacity = min(0.85, 0.2 + rel_flow * 0.65)
                
                curve_strength = min(0.3, 0.05 + 0.15 * rel_flow)
                
                angle_offset = (flow_count % 8) * (math.pi/4)
                offset_distance = 0.0001
                dx = math.cos(angle_offset) * offset_distance
                dy = math.sin(angle_offset) * offset_distance
                
                points = create_bezier_curve_with_strength(
                    start_station['lng'] + dx, start_station['lat'] + dy,
                    end_station['lng'] + dx, end_station['lat'] + dy,
                    curve_strength
                )
                
                popup_html = f"""
                <strong>From:</strong> {start_station['name']}<br>
                <strong>To:</strong> {end_station['name']}<br>
                <strong>Trips:</strong> {flow}
                """
                
                if flow >= top_threshold:
                    color = colormap(flow)
                    target_group = major_flows
                    width = min(width, 6.5)
                elif flow >= medium_threshold:
                    base_hue = station_colors.get(start_station['name'], 0)
                    saturation = 80 + rel_flow * 20
                    lightness = 50
                    color = f"hsl({base_hue}, {saturation}%, {lightness}%)"
                    target_group = medium_flows
                else:
                    base_hue = station_colors.get(start_station['name'], 0)
                    color = f"hsl({base_hue}, 70%, 60%)"
                    target_group = minor_flows
                    opacity = max(0.1, opacity - 0.1)
                
                folium.PolyLine(
                    points,
                    color=color,
                    weight=width,
                    opacity=opacity,
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=f"{start_station['name']} → {end_station['name']}: {flow} trips"
                ).add_to(target_group)
                
                flow_count += 1
    
    stations_group = folium.FeatureGroup(name="Stations", show=True)
    
    for _, station in stations_df.iterrows():
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=5,
            color='black',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(f"<strong>{station['name']}</strong>", max_width=300)
        ).add_to(stations_group)
    
    major_flows.add_to(m)
    medium_flows.add_to(m)
    minor_flows.add_to(m)
    stations_group.add_to(m)
    
    folium.LayerControl().add_to(m)
    
    return m

def create_time_period_flow_maps(trips_df, stations_df, station_mapping, output_dir="results"):
    """Create separate flow maps for each time period"""
    time_periods_dir = os.path.join(output_dir, "time_periods")
    os.makedirs(time_periods_dir, exist_ok=True)
    
    print("Generating flow maps and OD matrix visualizations...")
    
    for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
        print(f"Processing {period_name} with hours {start_hour}-{end_hour}")
        
        period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                              (trips_df['start_hour'] < end_hour)]
        
        print(f"Processing {period_name} with {len(period_trips)} trips")
        
        if len(period_trips) == 0:
            print(f"No trips found for {period_name}, skipping...")
            continue
            
        od_matrix = calculate_od_matrix(period_trips, station_mapping)
        
        flow_map = create_flow_map(
            od_matrix, 
            stations_df,
            title=f"Bike Flow Map - {period_name}",
            flow_scale=3.0,
            min_trips=5
        )
        
        safe_period_name = period_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "to")
        map_filename = f"flow_map_{safe_period_name}.html"
        flow_map.save(os.path.join(time_periods_dir, map_filename))
        print(f"Flow map for {period_name} saved to {os.path.join(time_periods_dir, map_filename)}")
        
        # Clear memory
        del period_trips, od_matrix, flow_map
        gc.collect()
    
    # Create overall flow map
    print("Creating overall flow map...")
    od_matrix_all = calculate_od_matrix(trips_df, station_mapping)
    all_day_map = create_flow_map(
        od_matrix_all, 
        stations_df,
        title="Bike Flow Map - All Day",
        flow_scale=3.0,
        min_trips=5
    )
    
    all_day_map.save(os.path.join(output_dir, "flow_map.html"))
    print(f"Flow map saved to {os.path.join(output_dir, 'flow_map.html')}")
    
    print("Flow visualizations completed")

###########################################
# Analysis Functions
###########################################

def analyze_seasonal_patterns(trips_df, output_dir):
    """Analyze seasonal patterns in Citibike trip data"""
    print("Analyzing seasonal patterns...")
    
    seasonal_dir = os.path.join(output_dir, 'seasonal_analysis')
    os.makedirs(seasonal_dir, exist_ok=True)
    
    seasons = {
        'winter': [1, 2, 12],
        'spring': [3, 4, 5],
        'summer': [6, 7, 8],
        'fall': [9, 10, 11]
    }
    
    if 'month' not in trips_df.columns:
        print("No month data available in trip data, extracting from datetime")
        if 'started_at' in trips_df.columns:
            trips_df['month'] = pd.to_datetime(trips_df['started_at']).dt.month
        else:
            print("Cannot analyze seasonal patterns: no month or datetime data available")
            return seasonal_dir
    
    available_months = trips_df['month'].unique()
    print(f"Available months: {sorted(available_months)}")
    
    monthly_trips = trips_df.groupby('month').size()
    
    seasonal_trips = {}
    for season_name, season_months in seasons.items():
        valid_months = [m for m in season_months if m in available_months]
        if valid_months:
            seasonal_trips[season_name] = monthly_trips[valid_months].sum()
    
    plt.figure(figsize=(12, 8))
    
    season_colors = {
        'winter': '#A1D6E2',
        'spring': '#81C784',
        'summer': '#FFB74D',
        'fall': '#E57373'
    }
    
    bars = plt.bar(
        range(len(seasonal_trips)),
        list(seasonal_trips.values()),
        width=0.7,
        color=[season_colors.get(s, '#BBBBBB') for s in seasonal_trips.keys()]
    )
    
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height * 1.02,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=12
        )
    
    plt.title('Citibike Trips by Season', fontsize=16, pad=20)
    plt.xlabel('Season', fontsize=14, labelpad=10)
    plt.ylabel('Number of Trips', fontsize=14, labelpad=10)
    plt.xticks(range(len(seasonal_trips)), [s.capitalize() for s in seasonal_trips.keys()], fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    seasonal_counts_path = os.path.join(seasonal_dir, 'seasonal_trip_counts.png')
    plt.savefig(seasonal_counts_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Seasonal analysis saved to {seasonal_counts_path}")
    return seasonal_dir

def analyze_day_type_patterns(trips_df, output_dir):
    """Analyze weekday vs weekend patterns in Citibike trip data"""
    print("Analyzing weekday vs weekend patterns...")
    
    day_type_dir = os.path.join(output_dir, 'day_type_analysis')
    os.makedirs(day_type_dir, exist_ok=True)
    
    if 'is_weekend' not in trips_df.columns and 'start_day' not in trips_df.columns:
        print("No day type data available in trip data, extracting from datetime")
        if 'started_at' in trips_df.columns:
            trips_df['start_day'] = pd.to_datetime(trips_df['started_at']).dt.dayofweek
            trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
        else:
            print("Cannot analyze day type patterns: no day or datetime data available")
            return day_type_dir
    
    if 'is_weekend' not in trips_df.columns and 'start_day' in trips_df.columns:
        trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    day_type_colors = {
        'weekday': '#3498db',
        'weekend': '#e74c3c'
    }
    
    day_type_counts = trips_df.groupby('is_weekend').size()
    day_type_labels = ['Weekday', 'Weekend']
    
    plt.figure(figsize=(10, 8))
    
    bars = plt.bar(
        day_type_labels,
        [day_type_counts.get(0, 0), day_type_counts.get(1, 0)],
        width=0.7,
        color=[day_type_colors['weekday'], day_type_colors['weekend']]
    )
    
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height * 1.02,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=14
        )
    
    plt.title('Citibike Trips: Weekday vs Weekend', fontsize=16, pad=20)
    plt.ylabel('Number of Trips', fontsize=14, labelpad=10)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(fontsize=14)
    
    day_type_counts_path = os.path.join(day_type_dir, 'day_type_trip_counts.png')
    plt.savefig(day_type_counts_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Day type analysis saved to {day_type_counts_path}")
    return day_type_dir

def create_od_matrix_heatmap(od_matrix, stations_df, period_name, output_dir):
    """Create heatmap visualization of OD matrix"""
    print(f"Creating OD matrix heatmap for {period_name}")
    
    plt.figure(figsize=(14, 12))
    
    # Create station name mapping for labels
    station_names = [stations_df.iloc[i]['name'][:20] + '...' if len(stations_df.iloc[i]['name']) > 20 
                    else stations_df.iloc[i]['name'] for i in range(len(stations_df))]
    
    # Create heatmap
    ax = sns.heatmap(
        od_matrix,
        cmap='YlOrRd',
        square=True,
        cbar_kws={'label': 'Number of Trips'},
        linewidths=0.1,
        linecolor='white',
        xticklabels=station_names,
        yticklabels=station_names
    )
    
    plt.title(f'Origin-Destination Matrix: {period_name}', fontsize=16, pad=20)
    plt.xlabel('Destination Station', fontsize=14, labelpad=10)
    plt.ylabel('Origin Station', fontsize=14, labelpad=10)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    
    # Save heatmap
    safe_name = period_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    heatmap_path = os.path.join(output_dir, f'{safe_name}_od_matrix_heatmap.png')
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"OD matrix heatmap saved to {heatmap_path}")
    return heatmap_path

###########################################
# Main Execution Functions
###########################################

def process_citibike_data(data_files_pattern, config=None, mode='all'):
    """Process Citibike data and run analysis with memory management"""
    if config is None:
        config = RUNTIME_CONFIG
    
    os.makedirs(config['OUTPUT_DIR'], exist_ok=True)
    
    print("Starting Citibike OD Analysis with memory management")
    print(f"Mode: {mode}")
    print(f"Max stations: {config['MAX_STATIONS']}")
    print(f"Sample rate: {config['SAMPLE_RATE']}")
    
    try:
        # Load data with memory management
        trips_df = load_citibike_data(
            config['DATA_DIR'], 
            config['DATA_FILES_PATTERN'],
            sample_rate=config['SAMPLE_RATE'],
            chunk_size=config['CHUNK_SIZE']
        )
        
        # Get top stations and create station mapping
        station_mapping, station_data, top_station_names, stations_df = get_top_stations(
            trips_df, config['MAX_STATIONS']
        )
        
        print(f"Working with {len(station_mapping)} stations")
        
        # Run visualizations if requested
        if mode in ['visualize', 'all']:
            print("\nGenerating visualizations...")
            
            # Create interactive station map
            interactive_station_map = create_interactive_station_map(
                stations_df,
                map_title="Citibike Stations",
                output_dir=config['OUTPUT_DIR']
            )
            
            # Create flow maps for different time periods
            create_time_period_flow_maps(trips_df, stations_df, station_mapping, config['OUTPUT_DIR'])
            
            # Create OD matrix heatmaps for each time period
            print("\nCreating OD matrix heatmaps...")
            heatmap_dir = os.path.join(config['OUTPUT_DIR'], 'od_matrix_heatmaps')
            os.makedirs(heatmap_dir, exist_ok=True)
            
            for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
                period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                                      (trips_df['start_hour'] < end_hour)]
                
                if len(period_trips) > 0:
                    od_matrix = calculate_od_matrix(period_trips, station_mapping)
                    create_od_matrix_heatmap(od_matrix, stations_df, period_name, heatmap_dir)
                    
                    # Clear memory
                    del period_trips, od_matrix
                    gc.collect()
            
            # Seasonal and temporal analysis
            if config.get('ANALYZE_SEASONAL_PATTERNS', True):
                analyze_seasonal_patterns(trips_df, config['OUTPUT_DIR'])
            
            analyze_day_type_patterns(trips_df, config['OUTPUT_DIR'])
            
            print("\nAll visualizations completed!")
        
        # Basic analysis and statistics
        print("\nGenerating summary statistics...")
        create_summary_report(trips_df, stations_df, station_mapping, config['OUTPUT_DIR'])
        
        print("\nAnalysis completed successfully!")
        return True
        
    except Exception as e:
        print(f"Error during analysis: {e}")
        import traceback
        traceback.print_exc()
        return False

def create_summary_report(trips_df, stations_df, station_mapping, output_dir):
    """Create a summary report of the analysis"""
    print("Creating summary report...")
    
    report_path = os.path.join(output_dir, 'analysis_summary.txt')
    
    # Calculate statistics
    total_trips = len(trips_df)
    total_stations = len(stations_df)
    date_range = f"{trips_df['started_at'].min().strftime('%Y-%m-%d')} to {trips_df['started_at'].max().strftime('%Y-%m-%d')}"
    
    # Top stations by popularity
    top_5_stations = stations_df.nlargest(5, 'popularity')
    
    # Peak hours
    hourly_trips = trips_df.groupby('start_hour').size()
    peak_hour = hourly_trips.idxmax()
    peak_hour_trips = hourly_trips.max()
    
    # Weekend vs weekday
    weekday_trips = len(trips_df[trips_df['is_weekend'] == 0])
    weekend_trips = len(trips_df[trips_df['is_weekend'] == 1])
    
    # Average trip duration
    avg_duration = trips_df['duration_minutes'].mean()
    median_duration = trips_df['duration_minutes'].median()
    
    # Use UTF-8 encoding to handle special characters properly
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("                    CITIBIKE ORIGIN-DESTINATION ANALYSIS SUMMARY\n")
        f.write("="*80 + "\n\n")
        
        f.write("DATASET OVERVIEW\n")
        f.write("-"*40 + "\n")
        f.write(f"Total Trips Analyzed: {total_trips:,}\n")
        f.write(f"Number of Stations: {total_stations}\n")
        f.write(f"Date Range: {date_range}\n")
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("TRIP CHARACTERISTICS\n")
        f.write("-"*40 + "\n")
        f.write(f"Average Trip Duration: {avg_duration:.1f} minutes\n")
        f.write(f"Median Trip Duration: {median_duration:.1f} minutes\n")
        f.write(f"Peak Hour: {peak_hour}:00 ({peak_hour_trips:,} trips)\n")
        f.write(f"Weekday Trips: {weekday_trips:,} ({weekday_trips/total_trips*100:.1f}%)\n")
        f.write(f"Weekend Trips: {weekend_trips:,} ({weekend_trips/total_trips*100:.1f}%)\n\n")
        
        f.write("TOP 5 STATIONS BY POPULARITY\n")
        f.write("-"*40 + "\n")
        for i, (_, station) in enumerate(top_5_stations.iterrows(), 1):
            f.write(f"{i}. {station['name']} ({station['popularity']:,} trips)\n")
        f.write("\n")
        
        f.write("TIME PERIOD ANALYSIS\n")
        f.write("-"*40 + "\n")
        for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
            if period_name != "All Day":
                period_trips = len(trips_df[(trips_df['start_hour'] >= start_hour) & 
                                          (trips_df['start_hour'] < end_hour)])
                percentage = period_trips / total_trips * 100
                f.write(f"{period_name}: {period_trips:,} trips ({percentage:.1f}%)\n")
        f.write("\n")
        
        f.write("GENERATED OUTPUTS\n")
        f.write("-"*40 + "\n")
        f.write("[+] Interactive station map (interactive_station_map.html)\n")
        f.write("[+] Flow maps for each time period (time_periods/)\n")
        f.write("[+] OD matrix heatmaps (od_matrix_heatmaps/)\n")
        f.write("[+] Seasonal analysis (seasonal_analysis/)\n")
        f.write("[+] Day type analysis (day_type_analysis/)\n")
        f.write("[+] Overall flow map (flow_map.html)\n\n")
        
        f.write("="*80 + "\n")
    
    print(f"Summary report saved to {report_path}")

def main():
    """Main entry point for Citibike OD Matrix Analysis"""
    print("Starting analysis with parameters:")
    print({
        'data_dir': RUNTIME_CONFIG['DATA_DIR'],
        'output_dir': RUNTIME_CONFIG['OUTPUT_DIR'],
        'max_stations': RUNTIME_CONFIG['MAX_STATIONS'],
        'sample_rate': RUNTIME_CONFIG['SAMPLE_RATE']
    })
    
    full_pattern = os.path.join(RUNTIME_CONFIG['DATA_DIR'], RUNTIME_CONFIG['DATA_FILES_PATTERN'])
    
    print(f"Data files pattern: {full_pattern}")
    print(f"Output directory: {RUNTIME_CONFIG['OUTPUT_DIR']}")
    print(f"Mode: {RUNTIME_CONFIG['MODE']}")
    
    try:
        success = process_citibike_data(full_pattern, RUNTIME_CONFIG, RUNTIME_CONFIG['MODE'])
        if success:
            print("\n[SUCCESS] Analysis completed successfully!")
            print(f"[FOLDER] Check results in: {RUNTIME_CONFIG['OUTPUT_DIR']}")
            print("[INFO] Open the HTML files in a web browser to view interactive maps")
        else:
            print("\n[ERROR] Analysis failed. Check error messages above.")
        return 0 if success else 1
    except FileNotFoundError as e:
        print(f"[ERROR] File not found: {e}")
        return 1
    except MemoryError as e:
        print(f"[ERROR] Memory error: {e}")
        print("[TIP] Try reducing SAMPLE_RATE or MAX_STATIONS in RUNTIME_CONFIG")
        return 1
    except Exception as e:
        print(f"[ERROR] Unexpected error: {e}")
        import traceback
        traceback.print_exc()
        return 1

if __name__ == "__main__":
    main()

Starting analysis with parameters:
{'data_dir': 'data/citibike', 'output_dir': 'results', 'max_stations': 50, 'sample_rate': 1.0}
Data files pattern: data/citibike\JC2024*citibiketripdata.csv
Output directory: results
Mode: all
Starting Citibike OD Analysis with memory management
Mode: all
Max stations: 50
Sample rate: 1.0
Loading Citibike data with 100.0% sampling rate
Found 12 files to process
Processing file: JC202401citibiketripdata.csv
Extracted month: 1
Loaded 50661 trips from JC202401citibiketripdata.csv
Processing file: JC202402citibiketripdata.csv
Extracted month: 2
Loaded 55613 trips from JC202402citibiketripdata.csv
Processing file: JC202403citibiketripdata.csv
Extracted month: 3
Loaded 65581 trips from JC202403citibiketripdata.csv
Processing file: JC202404citibiketripdata.csv
Extracted month: 4
Loaded 79116 trips from JC202404citibiketripdata.csv
Processing file: JC202405citibiketripdata.csv
Extracted month: 5
  Processed 10 chunks...
Loaded 97479 trips from JC202405citibik